# Data Analysis — Adult (Census Income) & Diabetes 130-US Hospitals
**Sarthak Shakya (s4238557), RMIT University**

Assignment: Part 1.3 — Data Analysis (Data Steward Analyst role)

Two models are applied to two datasets: **Logistic Regression** and **SVM (linear kernel)**.
Both are new to this assignment (Decision Tree and k-NN were used in Practical Data Science, COSC2670).

In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix, classification_report)

RANDOM_STATE = 42

## Dataset 1: Adult (Census Income)

Source: https://archive.ics.uci.edu/dataset/2/adult

48,842 records, 14 attributes. Target: whether income exceeds \$50K/year.

In [24]:
cols = ["age","workclass","fnlwgt","education","education_num","marital_status",
        "occupation","relationship","race","sex","capital_gain","capital_loss",
        "hours_per_week","native_country","income"]

train = pd.read_csv("adult.data", names=cols, skipinitialspace=True, na_values="?")
test = pd.read_csv("adult.test", names=cols, skipinitialspace=True, na_values="?", skiprows=1)
test["income"] = test["income"].str.replace(".", "", regex=False)

adult = pd.concat([train, test], ignore_index=True)
print(f"Total records: {len(adult)}")
print(f"Missing values per column:\n{adult.isna().sum()[adult.isna().sum()>0]}")
print(f"\nClass balance:\n{adult['income'].value_counts(normalize=True)}")

Total records: 48842
Missing values per column:
workclass         2799
occupation        2809
native_country     857
dtype: int64

Class balance:
income
<=50K    0.760718
>50K     0.239282
Name: proportion, dtype: float64


### Data quality handling
Rows with missing values in `workclass`, `occupation`, or `native_country` are dropped rather than imputed,
since these are categorical fields where imputation risks introducing bias — a data-governance-relevant
decision worth documenting explicitly.

In [25]:
n_before = len(adult)
adult_clean = adult.dropna()
print(f"Dropped {n_before - len(adult_clean)} rows with missing values ({(n_before-len(adult_clean))/n_before*100:.1f}%)")

# Keep race/sex aside for a fairness breakdown before encoding
race_sex = adult_clean[["race", "sex"]].reset_index(drop=True)

y = (adult_clean["income"] == ">50K").astype(int)
X = adult_clean.drop(columns=["income"])

cat_cols = X.select_dtypes(include="object").columns
le_dict = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c])
    le_dict[c] = le

X_train, X_test, y_train, y_test, rs_train, rs_test = train_test_split(
    X, y, race_sex, test_size=0.3, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

Dropped 3620 rows with missing values (7.4%)


### Model training: Logistic Regression and SVM

In [26]:
results_adult = {}

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
lr.fit(X_train_s, y_train)
pred_lr = lr.predict(X_test_s)
proba_lr = lr.predict_proba(X_test_s)[:, 1]
results_adult["Logistic Regression"] = {
    "accuracy": accuracy_score(y_test, pred_lr),
    "precision": precision_score(y_test, pred_lr),
    "recall": recall_score(y_test, pred_lr),
    "f1": f1_score(y_test, pred_lr),
    "roc_auc": roc_auc_score(y_test, proba_lr)
}

# SVM 
# Sample 8,000 observations from the training set for SVM
sample_idx = X_train.sample(
    n=8000,
    random_state=RANDOM_STATE
).index

X_train_svm = X_train.loc[sample_idx]
y_train_svm = y_train.loc[sample_idx]

# Scale the sampled training data
X_train_svm = scaler.transform(X_train_svm)

# Train SVM
svm_model = LinearSVC(
    random_state=RANDOM_STATE,
    max_iter=5000
)

svm_model.fit(X_train_svm, y_train_svm)
#make prediction on the test set
pred_svm = svm_model.predict(X_test_s)

results_adult["SVM"] = {
    "accuracy": accuracy_score(y_test, pred_svm),
    "precision": precision_score(y_test, pred_svm),
    "recall": recall_score(y_test, pred_svm),
    "f1": f1_score(y_test, pred_svm)
}

svm_scores = svm_model.decision_function(X_test_s)

results_adult["SVM"]["roc_auc"] = roc_auc_score(
    y_test,
    svm_scores
)

pd.DataFrame(results_adult).T

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.818235,0.709738,0.450922,0.551473,0.848068
SVM,0.817793,0.723394,0.428614,0.538289,0.847347


### Fairness breakdown by sex and race (Logistic Regression)

Since `race` and `sex` are sensitive attributes, checking whether model recall is consistent across
groups is a basic fairness audit which is relevant to the data steward role's responsibility for
reviewing sensitive data use.

In [27]:
fair_df = rs_test.reset_index(drop=True).copy()
fair_df["y_true"] = y_test.reset_index(drop=True)
fair_df["y_pred"] = pred_lr

print("By sex:")
for grp, sub in fair_df.groupby("sex"):
    r = recall_score(sub['y_true'], sub['y_pred']) if sub['y_true'].sum() > 0 else np.nan
    print(f"{grp}: n={len(sub)}, positive_rate_true={sub['y_true'].mean():.3f}, "
          f"positive_rate_pred={sub['y_pred'].mean():.3f}, recall={r:.3f}")

print("\nBy race:")
for grp, sub in fair_df.groupby("race"):
    if sub['y_true'].sum() > 0:
        r = recall_score(sub['y_true'], sub['y_pred'])
        print(f"{grp}: n={len(sub)}, positive_rate_true={sub['y_true'].mean():.3f}, "
              f"positive_rate_pred={sub['y_pred'].mean():.3f}, recall={r:.3f}")

By sex:
Female: n=4390, positive_rate_true=0.116, positive_rate_pred=0.034, recall=0.194
Male: n=9177, positive_rate_true=0.311, positive_rate_pred=0.217, recall=0.497

By race:
Amer-Indian-Eskimo: n=146, positive_rate_true=0.096, positive_rate_pred=0.041, recall=0.214
Asian-Pac-Islander: n=389, positive_rate_true=0.272, positive_rate_pred=0.149, recall=0.434
Black: n=1253, positive_rate_true=0.122, positive_rate_pred=0.055, recall=0.327
Other: n=99, positive_rate_true=0.152, positive_rate_pred=0.071, recall=0.400
White: n=11680, positive_rate_true=0.263, positive_rate_pred=0.171, recall=0.459


In [28]:
print("Top 10 coefficients for Logistic Regression Adult dataset")
coef_adult = pd.Series(lr.coef_[0], index=X.columns).abs().sort_values(ascending=False)
coef_adult.head(10)

Top 10 coefficients for Logistic Regression Adult dataset


capital_gain      2.463410
education_num     0.861334
age               0.472432
sex               0.441566
marital_status    0.339345
hours_per_week    0.332857
capital_loss      0.281247
relationship      0.221377
workclass         0.128642
race              0.088940
dtype: float64

## Dataset 2: Diabetes 130-US Hospitals (1999–2008)

Source: https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008

101,766 patient encounter records, 50 attributes. Target: hospital readmission within 30 days.

In [29]:
diab = pd.read_csv("diabetic_data.csv", na_values="?")
print(f"Total records: {len(diab)}")

n_unique_patients = diab["patient_nbr"].nunique()
print(f"Unique patients: {n_unique_patients} vs total visits: {len(diab)} "
      f"({len(diab)-n_unique_patients} repeat visit)")

Total records: 101766
Unique patients: 71518 vs total visits: 101766 (30248 repeat visit)


C:\Users\shaky\AppData\Local\Temp\ipykernel_1040\3314355637.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  diab = pd.read_csv("diabetic_data.csv", na_values="?")


### Data quality handling

- **Duplicate data:** the dataset contains multiple entry for the same patient. Only each
  patient's first entry is kept, to avoid data leakage from repeated patient history.
- **High-missing value columns** (`weight` 96%, `max_glu_serum` 95%, `A1Cresult` 82%) are dropped.
- **Moderately missing categorical fields** (e.g. `race`, diagnosis codes, `payer_code`,
  `medical_specialty`) are imputed as an explicit `"Unknown"` category rather than dropped, to avoid
  discarding most of the dataset — a deliberate completeness-vs-data-loss governance trade-off.

### Data Preparation and Methodology

The Diabetes 130-US Hospitals dataset was first sorted by `encounter_id` and duplicate patients were removed using `patient_nbr`, keeping the first recorded encounter for each patient. This prevents the same patient from appearing multiple times in the modelling dataset.

The target variable was created from `readmitted`, where patients readmitted within 30 days (`<30`) were assigned a value of 1 and all other patients were assigned a value of 0.

Missing categorical values were handled before model training. Categorical variables were encoded into numerical values so that they could be used by the machine learning models.

The data was then divided into training and testing sets. Feature scaling was applied using the training data to avoid data leakage. Because the target classes are imbalanced, `class_weight="balanced"` was used for the classification model so that the minority class received greater importance during training.

In [31]:
diab = diab.sort_values("encounter_id").drop_duplicates(subset="patient_nbr", keep="first")
print(f"After deduplication to first visit per patient: {len(diab)} records")

missing_pct = diab.isna().mean().sort_values(ascending=False)
print(f"Columns with >40% missing:\n{missing_pct[missing_pct>0.4]}")

diab = diab.drop(columns=[c for c in ["weight", "payer_code", "medical_specialty"] if c in diab.columns])

diab["target"] = (diab["readmitted"] == "<30").astype(int)
print(f"\nClass balance (re-admitted <30 days):\n{diab['target'].value_counts(normalize=True)}")

diab = diab.drop(columns=["readmitted", "encounter_id", "patient_nbr"])

for c in diab.select_dtypes(include="object").columns:
    diab[c] = diab[c].fillna("Unknown")
diab = diab.dropna()
print(f"After imputing missing categoricals as 'Unknown': {len(diab)} records")

After deduplication to first visit per patient: 71518 records
Columns with >40% missing:
max_glu_serum    0.951677
A1Cresult        0.818423
dtype: float64

Class balance (re-admitted <30 days):
target
0    0.912008
1    0.087992
Name: proportion, dtype: float64
After imputing missing categoricals as 'Unknown': 71518 records


In [32]:
y2 = diab["target"]
X2 = diab.drop(columns=["target"])

cat_cols2 = X2.select_dtypes(include="object").columns
for c in cat_cols2:
    le = LabelEncoder()
    X2[c] = le.fit_transform(X2[c].astype(str))

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.3, random_state=RANDOM_STATE, stratify=y2)

scaler2 = StandardScaler()
X2_train_s = scaler2.fit_transform(X2_train)
X2_test_s = scaler2.transform(X2_test)

### Model training

`class_weight="balanced"` is used for both models given the severe class imbalance
(~91%/9% split on the readmission target).

In [33]:
results_diab = {}

lr2 = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight="balanced")
lr2.fit(X2_train_s, y2_train)
pred_lr2 = lr2.predict(X2_test_s)
proba_lr2 = lr2.predict_proba(X2_test_s)[:, 1]
results_diab["Logistic Regression"] = {
    "accuracy": accuracy_score(y2_test, pred_lr2),
    "precision": precision_score(y2_test, pred_lr2),
    "recall": recall_score(y2_test, pred_lr2),
    "f1": f1_score(y2_test, pred_lr2),
    "roc_auc": roc_auc_score(y2_test, proba_lr2)
}

svm_n2 = min(8000, len(X2_train))
sample_idx2 = X2_train.sample(n=svm_n2, random_state=RANDOM_STATE).index
X2_train_svm = scaler2.transform(X2.loc[sample_idx2])
y2_train_svm = y2.loc[sample_idx2]

svm2 = SVC(kernel="linear", probability=True, random_state=RANDOM_STATE, class_weight="balanced")
svm2.fit(X2_train_svm, y2_train_svm)
pred_svm2 = svm2.predict(X2_test_s)
proba_svm2 = svm2.predict_proba(X2_test_s)[:, 1]
results_diab["SVM (linear)"] = {
    "accuracy": accuracy_score(y2_test, pred_svm2),
    "precision": precision_score(y2_test, pred_svm2),
    "recall": recall_score(y2_test, pred_svm2),
    "f1": f1_score(y2_test, pred_svm2),
    "roc_auc": roc_auc_score(y2_test, proba_svm2)
}

pd.DataFrame(results_diab).T

,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.633716,0.126766,0.537076,0.205118,0.622935
SVM (linear),0.716210,0.129998,0.390890,0.195109,0.600200


In [34]:
print("Top 10 coefficients for logistic Regression")
coef_diab = pd.Series(lr2.coef_[0], index=X2.columns).abs().sort_values(ascending=False)
coef_diab.head(10)

Top 10 coefficients for logistic Regression


number_inpatient            0.229937
discharge_disposition_id    0.153227
diabetesMed                 0.120720
age                         0.116311
time_in_hospital            0.107896
tolazamide                  0.105087
tolbutamide                 0.089798
chlorpropamide              0.061705
number_diagnoses            0.059195
number_emergency            0.053312
dtype: float64

## Summary of Results

| Dataset | Model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|---|
| Adult | Logistic Regression | 0.818 | 0.710 | 0.451 | 0.551 | 0.848 |
| Adult | SVM (linear) | 0.812 | 0.762 | 0.349 | 0.479 | 0.844 |
| Diabetes | Logistic Regression | 0.634 | 0.127 | 0.537 | 0.205 | 0.623 |
| Diabetes | SVM (linear) | 0.716 | 0.130 | 0.391 | 0.195 | 0.600 |

**Why not accuracy alone?** Both datasets are class-imbalanced (Adult ~76/24, Diabetes ~91/9), so a
naive majority-class classifier would already score 76–91% accuracy while giving no useful insight.
ROC-AUC (threshold-independent, imbalance-robust) is used as the primary comparison metric, with recall
weighted more heavily for the Diabetes case, since missing an at-risk patient (false negative) is more
costly than a false positive in a healthcare context.

**Insights are complementary, not contradictory:** the Adult dataset surfaces *fairness and
representation* risk (uneven recall across sex/race groups), while the Diabetes dataset surfaces
*data completeness and duplication* risk (extreme missingness, repeat encounters) and shows that
available administrative fields alone are insufficient to reliably predict readmission
(ROC-AUC ~0.60–0.62). Both reinforce the data steward role's core responsibility: surfacing these
issues before data or models are trusted by downstream users.